# **5.1 Layers and Blocks**
When we first introduced neural networks, we focused on linear models with a single output. Here, the entire model has only one output. Note that a single neural network (1) accepts some inputs; (2) generates a corresponding scalar output; and (3) has a set of associated parameters, the updating of which can optimise a certain objective function.<br><br>
Furthermore, when considering networks with multiple outputs, we utilise vectorised algorithms to describe an entire layer of neurons. Just like a single neuron, a layer (1) receives a set of inputs, (2) generates corresponding outputs, and (3) is described by a set of adjustable parameters. When using softmax regression, a single layer constitutes the model in itself. However, even if we subsequently introduce a multi-layer perceptron, we can still regard the model as retaining the basic architecture described above.<br><br>
In the case of a multi-layer perceptron, the entire model and its constituent layers follow this architecture. The model as a whole accepts raw inputs (features), generates outputs (predictions), and contains a number of parameters (the set of parameters for all constituent layers). Similarly, each individual layer receives an input (supplied by the previous layer), generates an output (the input to the next layer), and possesses a set of adjustable parameters, which are updated based on signals propagated backwards from the next layer.<br><br>
To realise these complex networks, we introduce the concept of neural network blocks. A block can describe a single layer, a component comprising multiple layers, or the entire model itself. One advantage of using blocks for abstraction is that several blocks can be combined to form larger components; this process is typically recursive, as shown in [Figure 5.1.1](#fig-511). By defining code to generate blocks of any complexity on demand, we can implement complex neural networks using concise code.
##### **(Fig 5.1.1)**
![large model created by blocks](../../images/picture5_1_1.png)
**From a programming perspective, a block is represented by a class. Any subclass of this class must define a forward propagation function that transforms its inputs into outputs, and must store any required parameters. Note that some blocks do not require any parameters. Finally, in order to compute gradients, a block must have a backpropagation function.** When defining our own blocks, as automatic differentiation (introduced in [Section 2.5](../../chapter_preliminaries/content/automatic_differentiation.ipynb#25-automatic-differentiation)) provides some backend implementations, we need only consider the forward propagation function and the required parameters.<br><br>
Before constructing a custom block, let us first review the code for the multi-layer perceptron ([Section 4.3](../../chapter_multi_layer_perceptron/content/concise_impl_of_mlp.ipynb#43-a-concise-implementation-of-a-multi-layer-perceptron)). The code below generates a network comprising a fully connected hidden layer with 256 units and a ReLU activation function, followed by a fully connected output layer with 10 hidden units and no activation function.

In [1]:
import torch
from torch import nn
from torch.nn import functional as F

net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))

X = torch.rand(2, 20)
net(X)

tensor([[ 0.1375,  0.2543, -0.0216, -0.2730,  0.2625,  0.1949,  0.1034,  0.0187,
         -0.2226,  0.0888],
        [ 0.1841,  0.1907,  0.0008, -0.2737,  0.3030,  0.1397,  0.0843,  0.0087,
         -0.2263,  0.1148]], grad_fn=<AddmmBackward0>)

In this example, we construct our model by instantiating `nn.Sequential`, with the execution order of the layers passed as a parameter. In short, `nn.Sequential` defines a special `Module`—the class in PyTorch that represents a block—which maintains an ordered list of `Module` instances. Note that both fully connected layers are instances of the `Linear` class, which is itself a subclass of `Module`. Furthermore, up to this point, we have been calling our model via `net(X)` to obtain its output. This is actually a shorthand for `net.__call__(X)`. This forward propagation function is very simple: it concatenates each block in the list, using the output of each block as the input to the next block.

## **5.1.1 Custom blocks**
The simplest way to gain an intuitive understanding of how blocks work is to implement one yourself. Before we implement our own custom block, let’s briefly summarize the basic functionality that every block must provide.
  1. Treat the input data as parameters for its forward propagation function.
  2. The output is generated via the forward propagation function. Please note that the shape of the output may differ from that of the input. For example, the first fully connected layer in our model above accepts a 20-dimensional input but returns an output of dimension 256.
  3. The gradient of the output with respect to the input can be accessed via its backpropagation function. This usually happens automatically.
  4. Store and access the parameters required for forward propagation calculations.
  5. Initialize the model parameters as required.

In the code snippet below, we write a block from scratch. It contains a multi-layer perceptron with a hidden layer comprising 256 hidden units and a 10-dimensional output layer. Note that the MLP class below inherits from the class representing the block. Our implementation only needs to provide our own constructor (the `__init__` function in Python) and forward propagation function.

In [2]:
class MLP(nn.Module):
    # Declare layers using model parameters. Here, we declare two fully connected layers
    def __init__(self):
        # Call the constructor of the MLP’s superclass, Module, to carry out the necessary initialization.
        # In this way, it is also possible to specify other function arguments when instantiating a class, such as the model parameter `params`.
        super().__init__()
        self.hidden = nn.Linear(20, 256) # the hidden layer
        self.out = nn.Linear(256, 10) # the output layer

    # Define the forward propagation of the model, i.e. how to return the desired model output based on the input X
    def forward(self, X):
        # Please note that here we are using the function version of ReLU, which is defined in the `nn.functional` module.
        return self.out(F.relu(self.hidden(X)))

Let us first look at the forward propagation function, which takes X as input, computes a hidden representation using an activation function, and outputs its unnormalised output values. In this MLP implementation, both layers are instance variables. To understand why this makes sense, imagine instantiating two multi-layer perceptrons (net1 and net2) and training them on different data sets. Naturally, we would expect them to learn two different models.<br><br>
Next, we instantiate the layers of the multi-layer perceptron, and then call these layers each time we invoke the forward propagation function. Note a few key details: firstly, our customised `__init__` function calls the parent class’s `__init__` function via `super().__init__()`, sparing us the hassle of rewriting boilerplate code. Then, we instantiate two fully connected layers, named `self.hidden` and `self.out` respectively. **Note that unless we implement a new operator, we need not concern ourselves with the backpropagation function or parameter initialization; the system will generate these automatically**.

In [3]:
net = MLP()
net(X)

tensor([[-0.0351, -0.0106, -0.0187, -0.0429,  0.0154,  0.0681,  0.0168, -0.0388,
          0.1487,  0.0804],
        [-0.0049, -0.0321, -0.1545, -0.1356, -0.0139,  0.0447,  0.0353, -0.0647,
          0.1195,  0.0872]], grad_fn=<AddmmBackward0>)

**One of the main advantages of a block is its versatility. We can subclass blocks to create layers (such as the fully connected layer class), entire models (such as the MLP class mentioned above), or various components of moderate complexity.** We make full use of this versatility in the following chapters, for example when dealing with convolutional neural networks.

## **5.1.2 Sequence block**
We can now take a closer look at how the `Sequential` class works. Recall that `Sequential` is designed to chain together other modules. To build our own simplified `MySequential`, we simply need to define two key functions:
  1. A function that appends blocks to a list one by one;
  2. A forward propagation function used to pass inputs through a ‘chain’ of blocks in the order in which the blocks were appended.


The `MySequential` class below provides the same functionality as the default `Sequential` class.

In [4]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            # In the variable _modules. The type of _module is OrderedDict.
            self._modules[str(idx)] = module

    def forward(self, X):
        # OrderedDict ensures that its elements are traversed in the order in which they were added
        for block in self._modules.values():
            X = block(X)
        return X

The `__init__` function adds each module one by one to the ordered dictionary _modules. **In short, the main advantage of _modules is that, during the initialization of a module’s parameters, the system knows to look in the _modules dictionary for the sub-blocks whose parameters need to be initialized.**<br><br>
When the forward propagation function of `MySequential` is called, each added block is executed in the order in which it was added. We can now use our `MySequential` class to re-implement a multi-layer perceptron.

In [5]:
net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X)

tensor([[-0.1565, -0.0915, -0.2394,  0.1343, -0.1097,  0.0296,  0.2232, -0.0611,
         -0.2194, -0.1528],
        [-0.0727, -0.1249, -0.1748,  0.0751, -0.0875, -0.0508,  0.3066, -0.0545,
         -0.1947, -0.1169]], grad_fn=<AddmmBackward0>)

## **5.1.3 Executing code in the forward propagation function**
The `Sequential` class simplifies model construction, allowing us to combine new architectures without having to define our own classes. However, not all architectures are simple sequential ones. When greater flexibility is required, we need to define our own blocks. For example, we may wish to execute Python control flow within the forward propagation function. Furthermore, we may wish to perform arbitrary mathematical operations, rather than simply relying on predefined neural network layers.<br><br>
So far, all operations in our network have acted upon the network’s activation values and parameters. However, there may be times when we wish to incorporate a term that is neither the result of the previous layer nor an updatable parameter; we refer to this as a constant parameter. For example, we require a layer that computes the function $f(x, w) = c \dot w^{T}x$, where $x$ is the input, w is a parameter, and c is a specified constant that is not updated during the optimization process. We have therefore implemented a `FixedHiddenMLP` class, as shown below:

In [6]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # Random weight parameters that are not used in gradient calculations. They therefore remain unchanged during training.
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)

    def forward(self, X):
        X = self.linear(X)
        # Use the constant parameters you have created, along with the relu and mm functions
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        # Reuse the fully connected layer. This is equivalent to two fully connected layers sharing parameters.
        X = self.linear(X)
        # Control Stream
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

In this `FixedHiddenMLP` model, we have implemented a hidden layer whose weights (`self.rand_weight`) are randomly initialized upon instantiation and remain constant thereafter. **These weights are not model parameters and are therefore never updated via backpropagation**. The neural network then passes the output of this fixed layer through a fully connected layer.<br><br>
Please note that, before returning the output, the model performs an unusual operation: it runs a `while` loop which, provided the L1 norm is greater than 1, divides the output vector by 2 until the condition is met. Finally, the model returns the sum of all terms in X. **Please note that this operation is unlikely to be commonly used in any practical task**; we are merely demonstrating how to integrate arbitrary code into the neural network computation process.

In [7]:
net = FixedHiddenMLP()
net(X)

tensor(-0.1804, grad_fn=<SumBackward0>)

We can mix and match various methods for combining blocks. In the examples below, we have nested blocks using a number of approaches that came to mind.

In [11]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(),
                                 nn.Linear(64, 32), nn.ReLU())
        self.linear = nn.Linear(32, 16)

    def forward(self, X):
        return self.linear(self.net(X))

chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP())
chimera(X)

tensor(0.1807, grad_fn=<SumBackward0>)

## **5.1.4 Efficiency**
Readers may begin to worry about operational efficiency. After all, we are performing a large number of dictionary lookups, code executions and many other Python operations within a high-performance deep learning library. The issue of the global interpreter lock (GIL) in Python is well known. In a deep learning environment, we are concerned that a high-speed GPU may have to wait for the CPU to finish executing Python code before it can run another job.

## **Summary**
  * A block may consist of many layers; a block may consist of many blocks.
  * Blocks may contain code.
  * The block is responsible for a large amount of internal processing, including parameter initialisation and backpropagation.
  * The sequential linking of layers and blocks is handled by the Sequential block.